# Chapter 09 — SQL Fundamentals and MySQL Connectivity

SQL is the language for relational data. Connect relational design and query logic to Python database drivers, transactions, parameterized SQL, and safe resource handling.

## Prerequisites
Basic tables and Python functions. MySQL server only needed for live-connection exercises; SQLite is used for local SQL drills.

## Learning goals
- Explain tables, rows, primary keys, and foreign keys.
- Write SELECT, WHERE, GROUP BY, HAVING, JOIN, and subqueries.
- Connect with mysql-connector-python or PyMySQL.
- Use parameterized statements, transactions, cursors, commits, and rollbacks.

**Study sequence:** use the theory notebook first, then tutorial, then lab. The examples use the bundled synthetic data unless otherwise noted.

In [1]:
from pathlib import Path
ROOT=Path.cwd()
if not (ROOT/"data").exists() and (ROOT.parent/"data").exists(): ROOT=ROOT.parent
DATA=ROOT/"data"
print("Data directory:", DATA.resolve())

Data directory: /mnt/data/python_data_analysis_book/data


## Guided learning path

Work from top to bottom. Each step states the purpose, explains the API choice, and provides a complete code cell. Modify parameters and predict the output before running it.

### Step 1 — Guided 1 — Read SQL rows

SQLite is included in Python, so it provides a local SQL engine without a separate server. MySQL uses similar relational ideas, though dialect details can differ.

In [2]:
import sqlite3
c=sqlite3.connect(":memory:")
c.executescript("CREATE TABLE teams(team_id INT PRIMARY KEY,name TEXT); CREATE TABLE scores(team_id INT,points INT);")
c.executemany("INSERT INTO teams VALUES (?,?)",[(1,"A"),(2,"B")]); c.executemany("INSERT INTO scores VALUES (?,?)",[(1,10),(1,20),(2,15)])
print(c.execute("SELECT * FROM scores").fetchall())

[(1, 10), (1, 20), (2, 15)]


### Step 2 — Guided 2 — Aggregate by group

`GROUP BY` creates groups; aggregate functions operate on each group. `HAVING` filters the aggregated result.

In [3]:
q="SELECT t.name,COUNT(*) AS n,SUM(s.points) AS total FROM teams t JOIN scores s USING(team_id) GROUP BY t.name HAVING SUM(s.points)>10 ORDER BY total DESC"
print(c.execute(q).fetchall())

[('A', 2, 30), ('B', 1, 15)]


### Step 3 — Guided 3 — Use parameters

SQLite placeholders use `?`; MySQL DB-API drivers commonly use `%s`. Values are separate from SQL code.

In [4]:
print(c.execute("SELECT * FROM scores WHERE points>?",(12,)).fetchall())

[(1, 20), (2, 15)]


### Step 4 — Guided 4 — Commit and rollback

A transaction is an atomic group of changes. Rollback cancels uncommitted work.

In [5]:
c.execute("UPDATE scores SET points=? WHERE team_id=?",(999,1)); c.rollback()
print(c.execute("SELECT points FROM scores WHERE team_id=1").fetchall()); c.close()

[]


## Tutorial checkpoint

1. What was the input data type?
2. What did the function or method return?
3. Which argument most changed the result?
4. What edge case should be tested before using this in production?